<a href="https://colab.research.google.com/github/ekrombouts/GenCareAI/blob/agitation_models/notebooks/200_comfort_score/210_CleanData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comfort score: Data cleaning and EDA

**Author:** Eva Rombouts  
**Date:** 2024-09-30  
**Version:** 0.1

### Description
This script performs data cleaning on a dataset of nursing home notes.
The cleaned dataset retains two columns: text and label. The label column is 1 if the category is 'symptomen' (symptoms) and 0 otherwise.

In [ ]:
!pip install GenCareAI
from GenCareAI.GenCareAIUtils import GenCareAISetup

setup = GenCareAISetup()

if setup.environment == 'Colab':
        !pip install -q datasets

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

In [ ]:
# Load the Hugging Face dataset
path_hf_dataset = 'ekrombouts/Gardenia_notes'
dataset = load_dataset(path_hf_dataset, token=setup.get_hf_token())
# Convert to pandas DataFrame
df = dataset['train'].to_pandas()
fn_cleaned = 'data/comfort_score/df_comfort.csv'

# Set parameters
word_count_max = 100

In [ ]:
#Quick Look
print(df.sample(5))
print(100*"*")
df.info()
print(100*"*")
print(df['category'].value_counts())

In [ ]:
# Check for rows with any missing values
missing_values = df[df.isna().any(axis=1)]
print(f"Number of rows with missing values: {len(missing_values)}")

# Suggestion: Create a new cell to inspect rows with missing values
if len(missing_values) > 0:
    print("\nRows with missing values:")
    print(missing_values)

In [ ]:
# Calculate word count of each note
df['note_word_count'] = df['note'].str.split().str.len()

# Plot histogram of word counts
plt.hist(df['note_word_count'], bins=100)
plt.title('Distribution of Word Counts in Notes')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.show()

# Print the 3 notes with the highest word count
top_3_longest_notes = df.nlargest(3, 'note_word_count')
print(top_3_longest_notes[['note_word_count', 'note']].to_string())


In [ ]:
# correlation between categories and note length
correlation = df[['category', 'note_word_count']].groupby('category').mean().round(1)
print("\nAverage word count by category:")
print(correlation)

In [ ]:
# Visualize the distribution of note lengths by category
df.boxplot(column='note_word_count', by='category', grid=False, figsize=(10, 6))
plt.title('Word Count Distribution by Category')
plt.suptitle('')
plt.show()

In [ ]:
# Clean the dataframe by removing missing values and large notes
df_cleaned = (
    df.dropna()  # Remove rows with missing values
      .drop(df[df['note_word_count'] > word_count_max].index)  # Remove rows with large notes
      .assign(label=(df['category'] == 'symptomen').astype(int))  # Create label column
      .rename(columns={'note': 'text'})  # Rename 'note' column to 'text'
      [['text', 'label']]  # Select only 'text' and 'label' columns
)

In [ ]:
# Show a random sample of 5 cleaned rows
print("Sample from cleaned data:")
print(df_cleaned.sample(5, ignore_index=True).to_string())
print(100*"*")

# Check label distribution
print("Label value counts:")
print(df_cleaned['label'].value_counts())
print(100*"*")
# Print label value counts as fractions
print(df_cleaned['label'].value_counts(normalize=True))
print(100*"*")
# Display cleaned dataset information
df_cleaned.info()


In [ ]:
# Investigate potential patterns in the text data (e.g., frequent words, n-grams)
from collections import Counter
from wordcloud import WordCloud

# Generate a word cloud for the 'symptomen' category
symptomen_notes = ' '.join(df_cleaned[df_cleaned['label'] == 1]['text'].tolist())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(symptomen_notes)

# Display the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud for the 'symptomen' Category", fontsize=16)
plt.show()

In [ ]:
# Create directory if it doesn't exist and save the cleaned dataset to CSV
os.makedirs(os.path.dirname(setup.get_file_path(fn_cleaned)), exist_ok=True)
df_cleaned.to_csv(setup.get_file_path(fn_cleaned), index=False)